# NYC Yellow Taxi 통계 분석 (2026-05)

본 노트북은 정제 완료된 뉴욕 택시 데이터를 기반으로 다양한 기초 기술통계 분석, 상관성 평가 및 과학적 가설 검정을 실시한 문서입니다.

### 분석 내용:
1. **기술통계 (Descriptive Statistics)**: 왜도(Skewness), 첨도(Kurtosis) 등을 추가해 데이터 분포 세부 양상 검토
2. **상관 분석 (Correlation Analysis)**: 변수 간 선형적 결합성 평가 및 최종 요금(`total_amount`)에 기여하는 변수 파악
3. **독립표본 t-검정 (t-test)**: 특정 요인(예: 평일 vs 주말, 낮 vs 밤)에 따른 요금 또는 팁의 통계적 평균 차이 검증

In [1]:
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats
import os
import json

### 데이터 불러오기

In [2]:
DATA_PATH = "../data/processed/yellow_tripdata_2026-05_clean.parquet"
df = pq.read_table(DATA_PATH).to_pandas()
print(f"[데이터 로드 완료] 행: {len(df):,}  열: {df.shape[1]}")

[데이터 로드 완료] 행: 2,877,997  열: 26


---
## 1. 기술통계 (Descriptive Statistics)

- 주요 수치형 컬럼들의 기초 통계 수치를 수집하고 대칭성(왜도) 및 극단값에 의한 뾰족함(첨도)을 산출합니다.
- 데이터 전처리 검증 차원에서 결측치 누락 여부를 최종 검증합니다.

In [3]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()

desc_stats = df[numeric_cols].describe().T
desc_stats["skew"] = df[numeric_cols].skew()
desc_stats["kurtosis"] = df[numeric_cols].kurt()

print("=" * 60)
print("수치형 변수 종합 기술통계량")
print("=" * 60)
display(desc_stats.round(3))

print("\n" + "=" * 60)
print("데이터셋 내 결측치 확인 (0명이어야 정상)")
print("=" * 60)
print(df.isnull().sum()[df.isnull().sum() > 0])

수치형 변수 종합 기술통계량


,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
VendorID,2877997.0,1.819,0.385,1.000,2.000,2.000,2.000,2.00,-1.657,0.747
passenger_count,2877997.0,1.258,0.643,1.000,1.000,1.000,1.000,6.00,3.179,12.046
trip_distance,2877997.0,3.189,4.247,0.010,1.000,1.660,3.100,97.78,3.096,13.493
RatecodeID,2877997.0,1.077,0.427,1.000,1.000,1.000,1.000,5.00,7.188,57.076
store_and_fwd_flag,2877997.0,0.001,0.034,0.000,0.000,0.000,0.000,1.00,29.521,869.508
PULocationID,2877997.0,167.266,62.977,1.000,132.000,162.000,234.000,265.00,-0.291,-0.810
DOLocationID,2877997.0,166.827,68.769,1.000,125.000,163.000,236.000,265.00,-0.408,-0.879
payment_type,2877997.0,1.139,0.385,1.000,1.000,1.000,1.000,4.00,3.233,13.542
fare_amount,2877997.0,19.598,17.935,0.010,9.300,13.500,21.900,888.00,3.580,32.862
extra,2877997.0,1.566,1.881,0.000,0.000,1.000,2.500,15.25,1.466,2.634



데이터셋 내 결측치 확인 (0명이어야 정상)
Series([], dtype: int64)


---
## 2. 상관분석 (Correlation Analysis)

- 피어슨(Pearson) 기법을 이용해 변수 쌍 간의 선형 상관관계를 확인합니다.
- 특히 최종 지불 총액(`total_amount`)과 높은 연관성을 띠는 핵심 피처 5개를 판별합니다.

In [4]:
corr_matrix = df[numeric_cols].corr(method="pearson")

print("=" * 60)
print("피어슨 상관계수 Matrix")
print("=" * 60)
display(corr_matrix.round(3))

print("\n" + "=" * 60)
print("total_amount와 상관관계가 높은 변수 Top 5 (자기 자신 제외)")
print("=" * 60)
top_corr = corr_matrix["total_amount"].drop("total_amount").abs().sort_values(ascending=False).head(5)
for col, corr_val in top_corr.items():
    raw_corr = corr_matrix.loc["total_amount", col]
    print(f"- {col:<25} : 상관계수 = {raw_corr:.3f}")

피어슨 상관계수 Matrix


,VendorID,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,...,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_min,average_speed_mph,pickup_hour,pickup_day_of_week,is_weekend
VendorID,1.000,0.099,0.036,0.029,-0.050,-0.020,-0.010,-0.017,0.051,-0.607,...,0.002,0.046,-0.028,0.040,0.003,0.023,0.043,0.031,0.011,0.016
passenger_count,0.099,1.000,0.048,0.082,0.001,-0.017,-0.011,0.018,0.062,-0.062,...,-0.001,0.057,-0.011,0.023,0.023,0.049,0.020,0.037,0.078,0.088
trip_distance,0.036,0.048,1.000,0.439,0.002,-0.141,-0.099,0.018,0.917,0.187,...,-0.001,0.916,-0.336,0.682,-0.060,0.795,0.678,-0.003,0.008,0.016
RatecodeID,0.029,0.082,0.439,1.000,0.001,-0.055,-0.021,0.010,0.587,-0.010,...,-0.003,0.560,-0.313,0.256,-0.067,0.301,0.346,-0.022,0.012,0.015
store_and_fwd_flag,-0.050,0.001,0.002,0.001,1.000,0.002,-0.001,0.001,0.003,0.030,...,-0.004,0.003,-0.001,-0.001,0.000,0.004,-0.001,0.001,-0.007,-0.003
PULocationID,-0.020,-0.017,-0.141,-0.055,0.002,1.000,0.072,-0.031,-0.131,-0.054,...,0.001,-0.133,0.140,-0.167,-0.146,-0.117,-0.114,0.004,-0.034,-0.038
DOLocationID,-0.010,-0.011,-0.099,-0.021,-0.001,0.072,1.000,-0.034,-0.097,-0.017,...,0.001,-0.093,0.139,-0.057,-0.112,-0.091,-0.086,0.025,-0.030,-0.032
payment_type,-0.017,0.018,0.018,0.010,0.001,-0.031,-0.034,1.000,0.014,-0.018,...,-0.003,-0.056,-0.130,0.063,-0.056,-0.005,0.038,-0.025,0.010,0.014
fare_amount,0.051,0.062,0.917,0.587,0.003,-0.131,-0.097,0.014,1.000,0.164,...,-0.001,0.980,-0.363,0.603,-0.021,0.833,0.555,0.001,-0.008,-0.010
extra,-0.607,-0.062,0.187,-0.010,0.030,-0.054,-0.017,-0.018,0.164,1.000,...,0.000,0.244,-0.056,0.319,0.003,0.173,0.154,0.167,-0.132,-0.172



total_amount와 상관관계가 높은 변수 Top 5 (자기 자신 제외)
- fare_amount               : 상관계수 = 0.980
- trip_distance             : 상관계수 = 0.916
- trip_duration_min         : 상관계수 = 0.830
- tip_amount                : 상관계수 = 0.761
- tolls_amount              : 상관계수 = 0.695


---
## 3. 독립표본 t-검정 (t-test)

- 독립된 두 표본 집단의 모평균이 같은지 다른지 통계적으로 검증합니다.
- 대량의 실제 데이터 분석이므로 두 집단의 분산이 다름을 가정하는 **Welch's t-test**(`equal_var=False`)를 적용합니다.

### 검정 체계:
- **귀무가설 ($H_0$)**: 두 그룹 간 평균에 통계적 차이가 존재하지 않는다.
- **대립가설 ($H_1$)**: 두 그룹 간 평균에 통계적으로 유의미한 차이가 존재한다.
- **유의수준 ($\alpha$)**: 0.05 (5%)

In [5]:
def run_ttest(group_col, value_col, group_a, group_b, alpha=0.05, hypothesis_label=None):
    """
    그룹 변수와 수치 변수를 입력받아 Welch's t-test를 자동 수행하고
    p-value 결과를 실무적 한글 문장으로 해설합니다.
    report.md 자동 생성을 위해 결과를 dict로도 반환합니다.
    """
    a = df[df[group_col] == group_a][value_col].dropna()
    b = df[df[group_col] == group_b][value_col].dropna()

    t_stat, p_value = stats.ttest_ind(a, b, equal_var=False)
    rejected = bool(p_value < alpha)

    print(f"\n[{value_col} 검정] {group_col} ({group_a}) vs {group_col} ({group_b})")
    print(f"  - 그룹 {group_a} 평균 : {a.mean():.3f} (N={len(a):,})")
    print(f"  - 그룹 {group_b} 평균 : {b.mean():.3f} (N={len(b):,})")
    print(f"  - t-statistic      : {t_stat:.4f}")
    print(f"  - p-value          : {p_value}")

    if rejected:
        conclusion = f"p-value({p_value:.3g}) < 유의수준({alpha})이므로 귀무가설을 기각합니다. 두 그룹의 '{value_col}' 평균에는 통계적으로 유의미한 차이가 존재합니다."
    else:
        conclusion = f"p-value({p_value:.3g}) >= 유의수준({alpha})이므로 귀무가설을 기각하지 못합니다. 두 그룹의 '{value_col}' 평균 차이는 통계적으로 유의하지 않습니다."
    print(f"  → 결과 해석: {conclusion}")

    return {
        "label": hypothesis_label or f"{group_col}: {group_a} vs {group_b} ({value_col})",
        "group_col": group_col,
        "value_col": value_col,
        "group_a": str(group_a),
        "group_b": str(group_b),
        "mean_a": float(a.mean()),
        "mean_b": float(b.mean()),
        "n_a": int(len(a)),
        "n_b": int(len(b)),
        "t_stat": float(t_stat),
        "p_value": float(p_value),
        "alpha": alpha,
        "rejected": rejected,
        "conclusion": conclusion,
    }

### t-test 실전 가설 검증

In [6]:
print("=" * 60)
print("독립표본 t-검정 결과 해석")
print("=" * 60)

ttest_results = []

# 가설 1: 평일(0) vs 주말(1)에 따른 지불 요금(total_amount) 평균 차이 검정
ttest_results.append(run_ttest(
    group_col="is_weekend", value_col="total_amount", group_a=0, group_b=1,
    hypothesis_label="평일 vs 주말 요금(total_amount)"
))

# 가설 2: 평일(0) vs 주말(1)에 따른 평균 이동 속도(average_speed_mph) 차이 검정
ttest_results.append(run_ttest(
    group_col="is_weekend", value_col="average_speed_mph", group_a=0, group_b=1,
    hypothesis_label="평일 vs 주말 평균 속도(average_speed_mph)"
))

# 가설 3: 낮(daytime) vs 심야(night) 시간대에 따른 팁 액수(tip_amount) 차이 검정
ttest_results.append(run_ttest(
    group_col="time_period", value_col="tip_amount", group_a="daytime", group_b="night",
    hypothesis_label="낮 vs 심야 평균 팁(tip_amount)"
))

독립표본 t-검정 결과 해석

[total_amount 검정] is_weekend (0) vs is_weekend (1)
  - 그룹 0 평균 : 30.082 (N=2,020,600)
  - 그룹 1 평균 : 28.726 (N=857,397)
  - t-statistic      : 45.9058
  - p-value          : 0.0
  → 결과 해석: p-value(0) < 유의수준(0.05)이므로 귀무가설을 기각합니다. 두 그룹의 'total_amount' 평균에는 통계적으로 유의미한 차이가 존재합니다.



[average_speed_mph 검정] is_weekend (0) vs is_weekend (1)
  - 그룹 0 평균 : 9.650 (N=2,020,600)
  - 그룹 1 평균 : 11.004 (N=857,397)
  - t-statistic      : -168.2026
  - p-value          : 0.0
  → 결과 해석: p-value(0) < 유의수준(0.05)이므로 귀무가설을 기각합니다. 두 그룹의 'average_speed_mph' 평균에는 통계적으로 유의미한 차이가 존재합니다.

[tip_amount 검정] time_period (daytime) vs time_period (night)
  - 그룹 daytime 평균 : 4.015 (N=1,352,780)
  - 그룹 night 평균 : 3.850 (N=473,992)
  - t-statistic      : 23.6262
  - p-value          : 2.2717512561707046e-123
  → 결과 해석: p-value(2.27e-123) < 유의수준(0.05)이므로 귀무가설을 기각합니다. 두 그룹의 'tip_amount' 평균에는 통계적으로 유의미한 차이가 존재합니다.


---
## 4. 통계 결과 리포트용 요약 내보내기
- 향후 Jinja2 템플릿 기반 마크다운 리포트 생성을 위해, 기술통계·상관계수 테이블은 `stat_summary.md`에,
  t-test 결과는 `ttest_results.json`에 저장합니다. `report.py`가 이 두 파일을 읽어 report.md의
  "1. 기술통계 및 상관관계 분석"과 "2. 가설 검정(t-test)" 섹션을 자동으로 채웁니다.

In [7]:
summary_lines = [
    "## 수치형 기술통계량 요약",
    "",
    desc_stats.round(3).to_markdown(),
    "",
    "## 상관계수 Matrix 요약",
    "",
    corr_matrix.round(3).to_markdown()
]

os.makedirs("../outputs", exist_ok=True)
with open("../outputs/stat_summary.md", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

with open("../outputs/ttest_results.json", "w", encoding="utf-8") as f:
    json.dump(ttest_results, f, ensure_ascii=False, indent=2)

print("[요약 내보내기 완료] outputs/stat_summary.md, outputs/ttest_results.json")

[요약 내보내기 완료] outputs/stat_summary.md, outputs/ttest_results.json
